# 打印知识树：读取 `batch_qa.ipynb` 建好的索引库，以树状形式打印

`batch_qa.ipynb` 的「步骤 2：建库」用 `superindex.md_ingest.index_markdown` 把每个 Markdown 切成一棵**章节树**
（节点字段：`title` / `node_id` / `start_index` / `end_index` / 可选 `summary` / 子节点 `nodes`），
落盘在 `<WORK_DIR>/store/docs/<doc_id>/tree.json`（`WORK_DIR` 默认 `<ROOT_DIR>/data/notebook/<题集名>/`）。

本 Notebook **只读取已落盘的树来打印，不重新建库、不调用任何模型**。先运行 `batch_qa.ipynb` 到步骤 2 即可；
工作目录的算法与它一致（`NB_WORK_DIR` / `DATASET_PATH` 环境变量，可写进 `ROOT_DIR/.env`）。

输出是**可折叠的 HTML 树**（`<details>` 原生折叠，不依赖 JS）：直接显示在 cell 输出里，同时导出一份**独立的 `tree.html`**
（写到 `<WORK_DIR>/tree.html`，双击即可在浏览器打开，带"全部展开 / 全部折叠"按钮，可直接发给别人）。
需要时也可以打开 `SHOW_TEXT_TREE` 再打印一份 `tree` 命令风格的纯文本树。

可以在 **Databricks** 上运行，也可以在**本地** Jupyter / VS Code / `nbconvert` 里运行（前置规则与 `batch_qa.ipynb` 一致）：

| | Databricks | 本地（仓库内） |
|---|---|---|
| 安装 | 运行「环境安装」两个 cell（`%pip install` + 重启 Python） | `uv sync --group notebook`；**不要运行**「环境安装」cell |
| 代码 | 在仓库里打开本 Notebook（Databricks 上先把仓库拉到最新）；「环境」cell 向上找仓库根并用其中的源码，找不到会报错 | 直接用当前仓库 |
| 数据 | 工作目录 `WORK_DIR` 与 `batch_qa.ipynb` 同一算法；树直接显示在 cell 输出里（用 `displayHTML`） | 同左（用 `display(HTML(...))`） |
| `tree.html` | 写到 `WORK_DIR`；设了 `PERSIST_DIR` 时额外复制一份到那里（持久位置，可下载） | 写到 `WORK_DIR` |

从上到下依次执行所有 cell。

## 环境安装（仅 Databricks）

**只在 Databricks 上运行下面两个 cell**（已打上 cell tag `databricks-setup`，与 `batch_qa.ipynb` 相同）。本地 Jupyter 已经 `uv sync --group notebook` 的，**不要运行**：`%pip install` 会把 PyPI 上的 `superindex` 装进当前环境，覆盖掉本地的可编辑安装。

- 第一个 cell 只放 `%pip install`（必须写在 cell 开头，才是 notebook 作用域安装）。`%pip` 是 notebook 作用域的，`batch_qa.ipynb` 里装的包本 Notebook 看不到，所以这里要自己装一次。
  只装 `superindex==0.1.2`：它自带 `python-dotenv`；打印用不到 pandas / openpyxl / pycryptodome，不装。集群访问不了公网 PyPI 时，换成 `batch_qa.ipynb` 里的备选写法（内部镜像 / Git URL / Repos / wheel），去掉后面的 `pandas openpyxl pycryptodome` 即可。
- 第二个 cell 重启 Python，让新装的包生效；只在 Databricks 上才真的重启。

**前提：先运行 `batch_qa.ipynb` 到步骤 2**（建库）。本 Notebook 不 git pull / clone，直接用所在仓库的代码：「环境」cell 向上找到仓库根并放到 `sys.path` 最前面，`ROOT_DIR` / `WORK_DIR` 因此与 `batch_qa.ipynb` 一致。
两个 Notebook 要在同一个仓库里运行（或设同样的 `ROOT_DIR` / `NB_WORK_DIR` 环境变量）。

命令行用 `nbconvert` 在本地执行时，用下面的参数跳过这两个 cell：

```bash
uv run --group notebook jupyter nbconvert --to notebook --execute notebooks/print_tree.ipynb \
  --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags databricks-setup \
  --output-dir <某目录>
```

In [ ]:
%pip install superindex==0.1.2

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

## 环境

确定 `ROOT_DIR`（环境变量 > 向上找到的仓库根；不在仓库里则报错）并加载 `.env`，与 `batch_qa.ipynb` 同一规则。

In [ ]:
# ───────────── 环境：确定 ROOT_DIR、加载 .env（与 batch_qa.ipynb 的"环境"cell 同一规则）─────────────
# ROOT_DIR：环境变量 ROOT_DIR > 从 cwd 向上找到的仓库根。本 Notebook 必须在仓库里运行（Databricks 上先把仓库拉到最新）。必须在 import superindex 之前做。
import html
import json
import os
import shutil
import sys
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, display


def find_repo_root(start: Path) -> Path | None:
    """向上找含 pyproject.toml 和 superindex/ 包目录的目录（即本仓库根）；找不到（pip 安装、不在仓库里）返回 None。"""
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "superindex").is_dir():
            return folder
    return None


IS_DATABRICKS = bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))
_repo_root = find_repo_root(Path.cwd().resolve())
if _repo_root is None:
    raise RuntimeError(
        f"从当前目录 {Path.cwd().resolve()} 向上找不到 SuperIndex 仓库根（含 pyproject.toml 和 superindex/）。"
        "请在仓库里打开并运行本 Notebook（Databricks 上先把仓库拉到最新）。"
    )
ROOT_DIR = Path(os.environ.get("ROOT_DIR", "").strip() or _repo_root).expanduser().resolve()
sys.path.insert(0, str(_repo_root))  # import superindex 用仓库里的源码，而不是 %pip 装的 PyPI 版；不 chdir


def under_root(p: str | Path) -> Path:
    """绝对路径原样（resolve）；相对路径以 ROOT_DIR 为基准。"""
    p = Path(p).expanduser()
    return (p if p.is_absolute() else ROOT_DIR / p).resolve()


# .env：先 ROOT_DIR/.env，再当前目录的 .env；都 override=False，所以进程环境变量优先
for _env_file in dict.fromkeys([ROOT_DIR / ".env", Path.cwd().resolve() / ".env"]):
    if _env_file.is_file():
        load_dotenv(_env_file, override=False, encoding="utf-8")
        print(f".env     : {_env_file} 已加载")
    else:
        print(f".env     : {_env_file} 不存在")
_root_from = "环境变量 ROOT_DIR" if os.environ.get("ROOT_DIR", "").strip() else "从当前目录向上找到的仓库根"
print(f"当前目录 : {Path.cwd().resolve()}")
print(f"ROOT_DIR : {ROOT_DIR}（来源：{_root_from}）")

from superindex.engine.local_store import DocStore

print(f"运行环境: {'Databricks' if IS_DATABRICKS else '本地'}")
print(f"superindex 来自: {Path(sys.modules['superindex'].__file__).resolve().parent}")


## 配置

工作目录的算法与 `batch_qa.ipynb` 一致；其余是打印选项。

In [ ]:
# ───────────── 配置（所有可调项都在这一格）─────────────
# 每项都可用同名环境变量覆盖（方便命令行 nbconvert 执行；也可写进 ROOT_DIR/.env）。
def _env(name: str, default, cast=str):
    raw = os.environ.get(name)
    return default if raw is None or raw.strip() == "" else cast(raw.strip())


# 工作目录：与 batch_qa.ipynb 完全一致 —— NB_WORK_DIR，否则 ROOT_DIR/data/notebook/<题集名>
DATASET_PATH = under_root(_env("DATASET_PATH", "data/questions.jsonl"))  # 只用来取题集名（文件不必存在）
WORK_DIR = under_root(os.environ.get("NB_WORK_DIR", "").strip() or ROOT_DIR / "data" / "notebook" / DATASET_PATH.stem)
STORE = WORK_DIR / "store"  # 索引库（batch_qa.ipynb 步骤 2 的产物）

DOC: str | None = _env("TREE_DOC", None)         # 只打印名字或 doc_id 含该子串的文档（不分大小写）；None = 全部
MAX_DEPTH: int | None = _env("TREE_MAX_DEPTH", None, int)  # 最多显示几层（文本树与 HTML 树都适用）（1 = 只打印顶层章节）；None = 不限
TEXT_WIDTH = _env("TREE_TEXT_WIDTH", 60, int)    # 节点标题 / 摘要各自最多显示多少字符，超出以 … 截断（HTML 里鼠标悬停可看全文）
OPEN_DEPTH: int | None = _env("TREE_OPEN_DEPTH", None, int)  # HTML 树默认展开前几层（1 = 只展开文档顶层章节列表；0 = 全部折叠）；None = 全部展开
SHOW_TEXT_TREE = _env("SHOW_TEXT_TREE", False, lambda t: t.lower() in ("1", "true", "yes", "on"))  # 另外打印纯文本树
HTML_PATH = WORK_DIR / "tree.html"               # 独立 html 的路径（与 batch_qa 的产物放在一起）
_persist = _env("PERSIST_DIR", None, Path)       # 与 batch_qa.ipynb 同名同义：持久目录；设了就把 tree.html 也复制一份过去
PERSIST_DIR: Path | None = under_root(_persist) if _persist else None
SHOW_PAGES = True                                # 节点后面显示页码范围 (p.起-止)
SHOW_SUMMARY = True                              # 节点有 summary 时显示（建库时 INDEX_SUMMARY=True 才有）

print(f"工作目录 : {WORK_DIR}   [{'存在' if WORK_DIR.is_dir() else '不存在'}]")
print(f"索引库   : {STORE}   [{'存在' if STORE.is_dir() else '不存在'}]")

## 打印知识树

用 `DocStore` 读每个文档的 `tree.json`，生成 HTML 树（cell 输出 + `tree.html`）；`SHOW_TEXT_TREE=True` 时另外打印纯文本连线树。

In [ ]:
# ───────────── 读取并打印（文本树 + HTML 树）─────────────
def clip(text: object, width: int) -> str:
    """压成单行并按 width 截断。"""
    s = " ".join(str(text or "").split())
    return s if len(s) <= width else s[: max(width - 1, 0)] + "…"


def count_nodes(nodes: list[dict]) -> int:
    return sum(1 + count_nodes(n.get("nodes") or []) for n in nodes)


def node_label(node: dict) -> str:
    """一个节点的显示文字：[node_id] 标题 (p.起-止) — 摘要。"""
    label = f"[{node.get('node_id', '?')}] {clip(node.get('title'), TEXT_WIDTH)}"
    if SHOW_PAGES and node.get("start_index") is not None:
        start, end = node["start_index"], node.get("end_index", node["start_index"])
        label += f" (p.{start})" if start == end else f" (p.{start}-{end})"
    if SHOW_SUMMARY and node.get("summary"):
        label += f" — {clip(node['summary'], TEXT_WIDTH)}"
    return label


def tree_lines(nodes: list[dict], prefix: str = "", depth: int = 1) -> list[str]:
    """tree 命令风格的连线；超过 MAX_DEPTH 的子树折叠成一行 '… (N 个节点)'。"""
    lines = []
    for i, node in enumerate(nodes):
        last = i == len(nodes) - 1
        lines.append(f"{prefix}{'└── ' if last else '├── '}{node_label(node)}")
        children = node.get("nodes") or []
        if not children:
            continue
        child_prefix = prefix + ("    " if last else "│   ")
        if MAX_DEPTH is not None and depth >= MAX_DEPTH:
            lines.append(f"{child_prefix}└── … （{count_nodes(children)} 个子节点未展开）")
        else:
            lines += tree_lines(children, child_prefix, depth + 1)
    return lines


store = DocStore(str(STORE))
docs = sorted((m for m in store.list_metas() if m.get("status") == "completed"), key=lambda m: m.get("name", ""))
if DOC:
    docs = [m for m in docs if DOC.lower() in m.get("name", "").lower() or DOC.lower() in m.get("id", "").lower()]
if not docs:
    raise RuntimeError(f"{STORE} 里没有可打印的文档（过滤条件 DOC={DOC!r}）。请先运行 batch_qa.ipynb 到步骤 2 建库，"
                       "或检查 NB_WORK_DIR / DATASET_PATH 是否与 batch_qa.ipynb 一致。")
trees = {m["id"]: store.get_tree(m["id"]) or [] for m in docs}

if SHOW_TEXT_TREE:
    print(f"共 {len(docs)} 个文档\n")
    for meta in docs:
        print(f"{meta.get('name')}  （doc_id {meta['id']}，{meta.get('pageNum')} 页，{count_nodes(trees[meta['id']])} 个节点）")
        print("\n".join(tree_lines(trees[meta["id"]])) or "  （空树）")
        print()


# ───────────── HTML 树：<details>/<summary> 嵌套；class 全部带 si- 前缀，不污染 notebook 其他输出 ─────────────
# 配色只用中性色 / 半透明灰 / 中等亮度的蓝，浅色和深色主题下都看得清；文字颜色继承所在环境。
TREE_CSS = """
.si-tree { font: 13px/1.55 -apple-system, "Segoe UI", "PingFang SC", "Microsoft YaHei", sans-serif; }
.si-tree .si-doc { margin: 0 0 18px; padding: 10px 14px; border: 1px solid rgba(128,128,128,.35); border-radius: 8px; }
.si-tree .si-doc-head { margin: 0 0 6px; font-size: 15px; font-weight: 600; }
.si-tree .si-meta { margin-left: 8px; font-size: 12px; font-weight: 400; opacity: .6; }
.si-tree details, .si-tree .si-leaf { margin: 0; }
.si-tree summary { cursor: pointer; list-style: none; }
.si-tree summary::-webkit-details-marker { display: none; }
.si-tree summary::before { content: "\\25B8"; display: inline-block; width: 16px; opacity: .55; }
.si-tree details[open] > summary::before { content: "\\25BE"; }
.si-tree .si-leaf { padding-left: 16px; }
.si-tree .si-row:hover { background: rgba(128,128,128,.12); }
.si-tree .si-children { margin-left: 7px; padding-left: 12px; border-left: 1px solid rgba(128,128,128,.35); }
.si-tree .si-id { font: 11px ui-monospace, Menlo, Consolas, monospace; opacity: .5; margin-right: 6px; }
.si-tree .si-title { font-weight: 500; }
.si-tree .si-pages { margin-left: 8px; font-size: 12px; color: #4f8fd9; white-space: nowrap; }
.si-tree .si-summary { margin-left: 8px; font-size: 12px; opacity: .65; }
.si-tree .si-more { font-size: 12px; font-style: italic; opacity: .5; }
"""


def node_row(node: dict) -> str:
    """一个节点的 HTML 行（所有来自数据的文本都转义）：node_id、标题、页码范围、摘要。"""
    title = " ".join(str(node.get("title") or "").split())
    parts = [f'<span class="si-id">{html.escape(str(node.get("node_id", "?")))}</span>',
             f'<span class="si-title" title="{html.escape(title, quote=True)}">{html.escape(clip(title, TEXT_WIDTH))}</span>']
    if SHOW_PAGES and node.get("start_index") is not None:
        start, end = node["start_index"], node.get("end_index", node["start_index"])
        parts.append(f'<span class="si-pages">p.{start}</span>' if start == end
                     else f'<span class="si-pages">p.{start}-{end}</span>')
    if SHOW_SUMMARY and node.get("summary"):
        full = " ".join(str(node["summary"]).split())
        parts.append(f'<span class="si-summary" title="{html.escape(full, quote=True)}">{html.escape(clip(full, TEXT_WIDTH))}</span>')
    return f'<span class="si-row">{"".join(parts)}</span>'


def node_html(node: dict, depth: int = 1) -> str:
    """叶子 = 不带箭头的普通行；有子节点 = <details>（depth <= OPEN_DEPTH 默认展开，其余折叠）。"""
    children = node.get("nodes") or []
    if not children:
        return f'<div class="si-leaf">{node_row(node)}</div>'
    if MAX_DEPTH is not None and depth >= MAX_DEPTH:
        inner = f'<div class="si-leaf si-more">… （{count_nodes(children)} 个子节点未展开）</div>'
    else:
        inner = "".join(node_html(c, depth + 1) for c in children)
    opened = " open" if OPEN_DEPTH is None or depth < OPEN_DEPTH else ""  # OPEN_DEPTH=2：顶层节点（depth 1）展开，其子节点们可见
    return f'<details{opened}><summary>{node_row(node)}</summary><div class="si-children">{inner}</div></details>'


def doc_html(meta: dict, tree: list[dict]) -> str:
    head = (f'<div class="si-doc-head">{html.escape(str(meta.get("name")))}'
            f'<span class="si-meta">doc_id {html.escape(str(meta["id"]))} · {html.escape(str(meta.get("pageNum")))} 页'
            f' · {count_nodes(tree)} 个节点</span></div>')
    body = "".join(node_html(n) for n in tree) or '<div class="si-leaf si-more">（空树）</div>'
    return f'<section class="si-doc">{head}{body}</section>'


tree_fragment = f'<style>{TREE_CSS}</style><div class="si-tree">{"".join(doc_html(m, trees[m["id"]]) for m in docs)}</div>'
if IS_DATABRICKS and "displayHTML" in globals():
    displayHTML(tree_fragment)  # noqa: F821 - displayHTML 由 Databricks 注入
else:
    display(HTML(tree_fragment))

# 独立 html：同一份片段 + 完整文档骨架 + "全部展开 / 全部折叠"按钮（少量内联 JS；没有 JS 时仍可逐个点开）
PAGE_TEMPLATE = """<!doctype html>
<html lang="zh-CN">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>知识树 · {title}</title>
<style>
body {{ margin: 0; padding: 24px; background: #fff; color: #1f2328; }}
@media (prefers-color-scheme: dark) {{ body {{ background: #14171c; color: #e6e8eb; }} }}
h1 {{ margin: 0 0 4px; font: 600 20px/1.4 -apple-system, "Segoe UI", "PingFang SC", "Microsoft YaHei", sans-serif; }}
.si-bar {{ margin: 0 0 16px; font: 12px/1.6 -apple-system, "Segoe UI", "PingFang SC", "Microsoft YaHei", sans-serif; opacity: .8; }}
.si-bar button {{ margin-right: 6px; padding: 2px 10px; cursor: pointer; font: inherit; }}
</style>
</head>
<body>
<h1>知识树</h1>
<div class="si-bar">
<button onclick="toggleAll(true)">全部展开</button><button onclick="toggleAll(false)">全部折叠</button>
生成时间 {stamp} · {n_docs} 个文档
</div>
{fragment}
<script>
function toggleAll(open) {{ document.querySelectorAll(".si-tree details").forEach(function (d) {{ d.open = open; }}); }}
</script>
</body>
</html>
"""
HTML_PATH.parent.mkdir(parents=True, exist_ok=True)
HTML_PATH.write_text(PAGE_TEMPLATE.format(title=html.escape(STORE.parent.name), stamp=datetime.now().strftime("%Y-%m-%d %H:%M"),
                                          n_docs=len(docs), fragment=tree_fragment), encoding="utf-8")
print(f"独立 html 已写出：{HTML_PATH}")
if PERSIST_DIR is not None and PERSIST_DIR.resolve() != WORK_DIR.resolve():  # 照搬 batch_qa 的 sync_to_persist：只复制内容，失败只警告
    try:
        PERSIST_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(HTML_PATH, PERSIST_DIR / HTML_PATH.name)
        print(f"已复制到 PERSIST_DIR：{PERSIST_DIR / HTML_PATH.name}")
    except Exception as exc:  # noqa: BLE001
        print(f"警告：复制到 PERSIST_DIR 失败（{type(exc).__name__}: {exc}）；本地文件仍在 {HTML_PATH}")
